# 🍏 Fun & Fit Health Advisor Agent Tutorial 🍎

Welcome to our **Fun & Fit Health Advisor Agent** tutorial, where you'll use **Azure AI Foundry** SDKs to create a playful (yet carefully disclaimed!) health and fitness assistant. We'll:

1. **Initialize** our project using **azure-ai-projects**.
2. **Create an Agent** specialized in providing general wellness and nutritional advice (with disclaimers!).
3. **Manage conversations** about fitness, nutrition, and general health topics.
4. **Note** that observability (logging and tracing with **OpenTelemetry**) is not covered here — it is demonstrated later in the dedicated Observability exercise.
5. **Demonstrate** how to incorporate tools, safety disclaimers, and basic best practices.

### ⚠️ Important Medical Disclaimer ⚠️
> **The health information provided by this notebook is for general educational and entertainment purposes only and is not intended as a substitute for professional medical advice, diagnosis, or treatment.** Always seek the advice of your physician or other qualified health provider with any questions you may have regarding a medical condition. Never disregard professional medical advice or delay seeking it because of something you read or receive from this notebook.


## Prerequisites

Complete [the notebooks in introduction](../../1-introduction/3-quick_start.ipynb)

## Let's Get Started
We'll walk you through each cell with notes and diagrams to keep it fun. Let's begin!

<img src="./seq-diagrams/1-basics.png" width="30%"/>



## 1. Initial Setup
We'll start by importing needed libraries, loading environment variables, and initializing an **AIProjectClient** so we can do all the agent-related actions. Let's do it! 🎉


In [ ]:
# Import required libraries
import os  # For environment variables and path operations
from pathlib import Path  # For cross-platform path handling

# Import Azure and utility libraries
from dotenv import load_dotenv  # For loading environment variables from .env file
from azure.identity import DefaultAzureCredential  # For Azure authentication
from azure.ai.projects import AIProjectClient  # Main client for AI Projects

# Get the path to the .env file which is in the parent directory
notebook_path = Path().absolute()  # Get absolute path of current notebook
parent_dir = notebook_path.parent.parent  # Get parent directory
load_dotenv(parent_dir / '.env')  # Load environment variables from .env file

# Initialize the endpoint-based AIProjectClient and its OpenAI (Responses API) client.
# The project client creates/versions agents; the OpenAI client runs conversations & responses.
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully initialized AIProjectClient + OpenAI client")
except Exception as e:
    # Print error message if client initialization fails
    print(f"❌ Error initializing project client: {str(e)}")

## 2. Creating our Fun & Fit Health Advisor Agent 🏋️

We'll create an Agent specialized in general health and wellness. We'll explicitly mention disclaimers in its instructions, so it never forgets to keep it safe! The instructions also ask the agent to focus on general fitness, dietary tips, and always encourage the user to seek professional advice.


In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

def create_health_advisor_agent():
    """Create a health advisor agent with disclaimers and basic instructions."""
    try:
        # Get the model name from environment variables, default to gpt-5.4 if not set
        model_name = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4")

        # Create a new agent version using a PromptAgentDefinition.
        # The definition holds the model + instructions that shape the agent's behavior.
        agent = project_client.agents.create_version(
            agent_name="fun-fit-health-advisor",
            definition=PromptAgentDefinition(
                model=model_name,
                instructions="""
                You are a friendly AI Health Advisor.
                You provide general health, fitness, and nutrition information, but always:
                1. Include medical disclaimers.
                2. Encourage the user to consult healthcare professionals.
                3. Provide general, non-diagnostic advice around wellness, diet, and fitness.
                4. Clearly remind them you're not a doctor.
                5. Encourage safe and balanced approaches to exercise and nutrition.
                """,
            ),
            description="Friendly health & fitness advisor with medical disclaimers.",
        )
        # Log success and return the created agent
        print(f"🎉 Created health advisor agent '{agent.name}', version: {agent.version}")
        return agent
    except Exception as e:
        # Handle any errors during agent creation
        print(f"❌ Error creating agent: {str(e)}")
        return None

# Create an instance of our health advisor agent
health_advisor = create_health_advisor_agent()

## 3. Managing Our Health Conversations 💬
A **conversation** is where we'll store the user's messages and the agent's responses about health topics (the v2 API uses conversations instead of threads). Let's create a new conversation dedicated to Health & Fitness Q&A.


In [ ]:
# Function to create a new conversation for health discussions
def start_health_conversation():
    """Create a new conversation for health & fitness discussions."""
    try:
        # Create a new empty conversation using the OpenAI (Responses API) client.
        # A conversation stores the back-and-forth items between user and agent.
        conversation = openai_client.conversations.create()

        # Print success message with the conversation ID for reference
        print(f"📝 Created a new conversation, ID: {conversation.id}")

        # Return the created conversation object so we can use it later
        return conversation
    except Exception as e:
        # If conversation creation fails, print error and return None
        print(f"❌ Error creating conversation: {str(e)}")
        return None

# Initialize a new conversation that we'll use for our health Q&A
health_conversation = start_health_conversation()

## 4. Asking Health & Fitness Questions 🏃‍♂️
We'll create messages from the user about typical health questions. For example, **"How do I calculate my BMI?"** or **"What's a balanced meal for an active lifestyle?"**. We'll let our Health Advisor Agent respond, always remembering that disclaimer!


In [ ]:
def ask_health_question(agent, conversation_id, user_question):
    """Send a user question to the agent inside a conversation, via the Responses API.

    Args:
        agent: the agent version returned by create_version
        conversation_id: ID of the conversation to keep context across turns
        user_question: The health/fitness question from the user

    Returns:
        The response object if successful, None if an error occurs
    """
    try:
        # A single responses.create call sends the user input, lets the agent run,
        # and appends both the question and answer to the conversation.
        response = openai_client.responses.create(
            conversation=conversation_id,
            input=user_question,
            extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        )
        return response
    except Exception as e:
        print(f"❌ Error asking question: {e}")
        return None

def print_response(user_question, response):
    """Print the user's question and the agent's grounded answer."""
    if response is None:
        return
    print(f"USER: {user_question}\n")
    print(f"ASSISTANT: {response.output_text}\n")
    print("-----------------------------------\n")

### Example Queries
Let's do some quick queries now to see the agent's disclaimers and how it handles typical health questions. We'll ask about **BMI** and about **balanced meal** for an active lifestyle.


In [ ]:
# First verify that we have valid agent and conversation objects before proceeding
if health_advisor and health_conversation:
    # 1) Ask about BMI calculation and interpretation
    # This demonstrates how the agent handles technical health questions
    q1 = "How do I calculate my BMI, and what does it mean?"
    r1 = ask_health_question(health_advisor, health_conversation.id, q1)
    print_response(q1, r1)

    # 2) Ask about personalized meal planning (the conversation keeps prior context)
    q2 = "Can you give me a balanced meal plan for someone who exercises 3x a week?"
    r2 = ask_health_question(health_advisor, health_conversation.id, q2)
    print_response(q2, r2)
else:
    # Error handling if agent or conversation initialization failed
    print("❌ Could not run example queries because agent or conversation is None.")

## 5. Cleanup 🧹
If you'd like to remove your agent from the service once finished, you can do so below. (In production, you might keep your agent around for stateful experiences!)

### 👀 See your agent live in the Foundry portal

Before deleting the agent, confirm it now exists as a first-class resource in your Foundry project:

1. In a browser, open the [Microsoft Foundry portal](https://ai.azure.com) and select your project.
2. In the top navigation select **Build**, then select **Agents** in the left pane.
3. Locate **`fun-fit-health-advisor`** in the list — this is the very agent you just created from code. Open it to inspect its instructions and tools.
4. Return to this notebook and run the next cell to delete the agent and free up resources.

> **Note:** Because several notebooks each create an agent, your project can accumulate them quickly — so every notebook deletes its own agent at the end.

In [ ]:
# Function to clean up and delete the agent version when we're done
def cleanup(agent):
    # Only attempt cleanup if we have a valid agent
    if agent:
        try:
            # Delete the agent version using the project client
            project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
            # Print confirmation message with the agent name
            print(f"🗑️ Deleted health advisor agent: {agent.name} (v{agent.version})")
        except Exception as e:
            # Handle any errors that occur during deletion
            print(f"Error cleaning up agent: {e}")
    else:
        # If no agent was provided, inform the user
        print("No agent to clean up.")

# Call cleanup function to delete our health advisor agent
cleanup(health_advisor)

# Congratulations! 🏆
You've successfully built a **Fun & Fit Health Advisor** that can:
1. **Respond** to basic health and fitness questions.
2. **Use disclaimers** to encourage safe, professional consultation.
3. **Provide** general diet and wellness information.
4. **Use** the synergy of **Azure AI Foundry** modules to power the conversation.

## Next Steps
- Explore adding more advanced tools (like **FileSearchTool** or **CodeInterpreterTool**) to provide more specialized info.
- Evaluate your AI's performance with **azure-ai-evaluation**!
- Add observability with **OpenTelemetry** or Azure Monitor for deeper insights — this is covered in the dedicated Observability exercise.
- Incorporate **function calling** if you want to handle things like advanced calculation or direct data analysis.

#### Let's proceed to [2-code_interpreter.ipynb](2-code_interpreter.ipynb)

Happy (healthy) coding! 💪